# Assignment NLP – 3: Chatbot using Transformers

## Task 1: Model Loading

### 1. Install Libraries

In [1]:
!pip install --upgrade transformers huggingface_hub accelerate

### 2. Importing required Libraries

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

### 3. Load Pre-Trained Model

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## Task 2: User Handling

Here Chatbot accepts input from user. This allows chatbot to start conversation with the user

In [4]:
# Taking input from user
user_input = input("You: ")
print(f"User Entered: {user_input}")

You: Hello
User Entered: Hello


## Task: 3: Response Generation (Transformer Logic)

In [8]:
def Generate_response(user_input):
        text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)

        # Converting text into model input tensors
        model_inputs = tokenizer([text],return_tensors="pt").to(model.device)

        # Generating response tokens from the model
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

        # Extracting only newly generated tokens (excluding input tokens)
        generated_ids = [output_ids[len(input_ids):]for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

        # Decoding tokens into readable text
        response = tokenizer.batch_decode(generated_ids,skip_special_tokens=True)[0]

        return response

## Task 4: Continuous Conversation

In [9]:
# Starting chatbot interaction loop
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

# Initializing conversation history with system instruction
messages = [
    {"role": "system", "content": "You are a helpful and friendly AI assistant."}
]

while True:
    # Taking user input
    user_input = input("You: ")

    # Checking for exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Adding user message to conversation history
    messages.append({
        "role": "user",
        "content": user_input
    })

    # Generating chatbot response based on conversation
    response = Generate_response(messages)

    print("Chatbot:", response)
    # Storing chatbot reply for maintaining context
    messages.append({
        "role": "assistant",
        "content": response
    })

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: Hello
Chatbot: Hello! How can I help you today? Do you have any specific questions or topics in mind that you'd like to discuss? I'm here to assist with any inquiries you might have, whether it's about science, technology, history, literature, culture, or anything else. Just let me know how I can be of assistance!
You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the creation of computer systems that exhibit intelligent behavior, such as learning, problem-solving, decision-making, and self-correction without being explicitly programmed. These systems are designed to perform tasks that would typically require human intelligence, such as recognizing speech patterns, understanding language, making decisions, playing games, controlling machines, and even interacting with humans.

Some key aspects of AI
You: Who created Python?
Chatbot: Python was created by Guido van Rossum at th

## Task 5: Exit Condition

By Entering "Exit" or 'Quit" the flow breaks and ends the conversation with the chatbot